## 13.09 用于预训练BERT的数据集


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()
import os
import random
import nltk
nltk.download('punkt_tab', quiet=True)
from src.utils import (download_extract, get_tokens_and_segments,
                       load_data_wiki)


### 练习 13.9.1

**题目：** 为简单起见，句号用作拆分句子的唯一分隔符。尝试其他的句子拆分技术，比如 Spacy 和 NLTK。以 NLTK 为例，需要先安装 NLTK：`pip install nltk`。在代码中先 `import nltk`。然后下载 Punkt 语句词元分析器。

**解答：** 主 notebook 的 `_read_wiki` 用 `' . '`（两边带空格的点）做句子分隔符，对带省略号、缩写（如 “Mr.”）或 `!`/`?` 的句子会失效。NLTK 的 `sent_tokenize`（Punkt）内置了缩略词表与句末标点规则，拆分更准确。下面用两个示例对比：

1. 简单感叹句（主 notebook 的方法也能处理）；
2. 含缩写 “Mr.” 与多个句末标点的句子（只有 NLTK 能正确拆分）。


In [2]:
from nltk.tokenize import sent_tokenize

for s in ['This is great ! Why not ?',
          'Mr. Smith went to the store. He bought milk! Really?']:
    print(f'原文: {s}')
    print(f'  split(" . ") : {s.strip().lower().split(" . ")}')
    print(f'  NLTK 分词  : {sent_tokenize(s.lower())}')


原文: This is great ! Why not ?
  split(" . ") : ['this is great ! why not ?']
  NLTK 分词  : ['this is great !', 'why not ?']
原文: Mr. Smith went to the store. He bought milk! Really?
  split(" . ") : ['mr. smith went to the store. he bought milk! really?']
  NLTK 分词  : ['mr. smith went to the store.', 'he bought milk!', 'really?']


进一步在 WikiText-2 真实数据上对比两种拆分得到的段落/句子统计（加载约需下载 13MB 数据）：


In [3]:
# 用 WikiText-2 前若干行对比两种拆分的句子数
data_dir = download_extract('wikitext-2', 'wikitext-2')
with open(os.path.join(data_dir, 'wiki.train.tokens')) as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]

lines = lines[:200]
n_dot = sum(len(l.split(' . ')) for l in lines)
n_nltk = sum(len(sent_tokenize(l)) for l in lines)
print(f'前 {len(lines)} 行：split(" . ") 拆分出 {n_dot} 句, '
      f'NLTK 拆分出 {n_nltk} 句（差异主要由缩写/多标点引起）')


正在从 HF 镜像下载 wikitext-2 parquet 并重建 zip...


wikitext-2 已重建：../data/wikitext-2-v1.zip（train 36717 行）
前 200 行：split(" . ") 拆分出 494 句, NLTK 拆分出 491 句（差异主要由缩写/多标点引起）


### 练习 13.9.2

**题目：** 如果我们不过滤出一些不常见的词元，词量会有多大？

**解答：** 主 notebook 的 `_WikiTextDataset` 在构建 `Vocab` 时使用 `min_freq=5`，把出现次数少于 5 次的低频词元统一映射为 `<unk>`，因此词表较小（约 2 万）。若把 `min_freq` 改为 0（不过滤任何词元），词表会膨胀到接近语料中去重后的全部词元数。下面用 `_WikiTextDataset` 分别以 `min_freq=5` 与 `min_freq=0` 统计词表大小：


In [4]:
from src.utils import _read_wiki, Vocab

data_dir = download_extract('wikitext-2', 'wikitext-2')
paragraphs = _read_wiki(data_dir)

from src.utils import tokenize

# 与 _WikiTextDataset 相同的 vocab 构建方式：先按句子分词，再统计词频
sentences = [sentence for para in paragraphs for sentence in para]
tokens = [tokenize(para, token='word') for para in paragraphs]
sentences = [sentence for para in tokens for sentence in para]

def vocab_size_for(min_freq):
    vocab = Vocab(sentences, min_freq=min_freq)
    return len(vocab)

print(f'min_freq=5（主 notebook 默认）: vocab size = {vocab_size_for(5)}')
print(f'min_freq=0（不过滤低频词）  : vocab size = {vocab_size_for(0)}')
print('低频词元被过滤后统一归为 <unk>，词表越小训练越省内存，但 OOV 损失越多。')


正在从 HF 镜像下载 wikitext-2 parquet 并重建 zip...


wikitext-2 已重建：../data/wikitext-2-v1.zip（train 36717 行）


min_freq=5（主 notebook 默认）: vocab size = 20252


min_freq=0（不过滤低频词）  : vocab size = 28882
低频词元被过滤后统一归为 <unk>，词表越小训练越省内存，但 OOV 损失越多。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
